# Stage 1 — V1 VideoMAEv2-B

Colab setup: GitHub code is expected under `/content/Blackbox-Detection`; only `DATASET/` is mounted from Google Drive. The team-provided `train.csv` / `val.csv` are used directly. `test.csv` is reserved for final holdout evaluation.

## 1. Setup

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

# Colab: mount only Google Drive data. The code repository remains under /content.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; Drive mount skipped.')


def find_repo_root() -> Path:
    candidates = [Path('/content/Blackbox-Detection'), Path.cwd()]
    for candidate in candidates:
        current = candidate.resolve()
        while True:
            if (current / 'pyproject.toml').is_file():
                return current
            if current == current.parent:
                break
            current = current.parent
    raise FileNotFoundError(
        'Blackbox-Detection repository not found. Clone/check out your GitHub repo first, '
        'normally at /content/Blackbox-Detection.'
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from blackbox_detection.utils import seed_everything, setup_logger

print('repo :', REPO_ROOT)
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
from blackbox_detection.stage1.dataset import Stage1VideoDataset, build_dataloader, video_batch_adapter
from blackbox_detection.stage1.evaluator import AggregationConfig, Stage1Evaluator, save_predictions
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_clip_sampler
from blackbox_detection.stage1.trainer import Stage1Trainer, TrainConfig
from blackbox_detection.stage1.transforms import ClipAugmentConfig, build_video_transforms
from blackbox_detection.utils import load_checkpoint

logger = setup_logger("stage1.videomaev2_b")

## 2. Paths

In [ ]:
CONFIG_DIR = REPO_ROOT / 'configs' / 'stage1'
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'stage1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Only the DATASET directory comes from Google Drive.
# If your Drive folder is elsewhere, edit this one line only.
DATASET_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection/DATASET')

# Preferred location for the team-provided fixed Stage 1 split CSVs.
# For convenience, DATASET_ROOT/train.csv etc. is also accepted.
preferred_split_root = DATASET_ROOT / 'stage1_splits'
root_split_files = [DATASET_ROOT / name for name in ('train.csv', 'val.csv', 'test.csv')]
if preferred_split_root.is_dir():
    SPLIT_ROOT = preferred_split_root
elif all(path.is_file() for path in root_split_files):
    SPLIT_ROOT = DATASET_ROOT
else:
    SPLIT_ROOT = preferred_split_root

TRAIN_CSV = SPLIT_ROOT / 'train.csv'
VAL_CSV = SPLIT_ROOT / 'val.csv'
TEST_CSV = SPLIT_ROOT / 'test.csv'  # final internal holdout; never used for tuning here

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f'DATASET_ROOT does not exist: {DATASET_ROOT}\n'
        'Mount Drive first or edit DATASET_ROOT.'
    )

print('dataset:', DATASET_ROOT)
print('splits :', SPLIT_ROOT)
print('train  :', TRAIN_CSV)
print('val    :', VAL_CSV)
print('test   :', TEST_CSV, '(reserved; not loaded)')

## 3. Config

In [ ]:
CONFIG = yaml.safe_load((CONFIG_DIR / 'videomaev2_b.yaml').read_text(encoding='utf-8'))
MODEL_NAME = CONFIG['model']['name']
ADAPTER = video_batch_adapter()
SEED = int(CONFIG['train']['seed'])
RUN_DIR = REPO_ROOT / CONFIG['train']['output_dir']
RUN_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(SEED, deterministic=False)
print(MODEL_NAME, '->', RUN_DIR)
print(json.dumps(CONFIG['model'], indent=2))

## 4. Fixed CSV data

In [ ]:
LABEL_ALIASES = {
    '0': 'ORIGINAL',
    'OR': 'ORIGINAL',
    'ORIGINAL': 'ORIGINAL',
    '1': 'RERECORDED',
    'RE': 'RERECORDED',
    'RERECORDED': 'RERECORDED',
    'RE-RECORDED': 'RERECORDED',
}


def _infer_dataset(path_text: str) -> str:
    lowered = path_text.replace('\\', '/').lower()
    if 'dlc-2021' in lowered or 'dlc2021' in lowered:
        return 'dlc2021'
    if '/ccd/' in f'/{lowered.strip("/")}/' or lowered.startswith('ccd/'):
        return 'ccd'
    return 'unknown'


def _stable_video_id(path_text: str) -> str:
    normalized = path_text.replace('\\', '/')
    stem = Path(normalized).stem
    digest = hashlib.sha1(normalized.encode('utf-8')).hexdigest()[:10]
    return f'{stem}_{digest}'


def _resolve_video_path(value: str) -> str:
    raw = str(value).strip()
    p = Path(raw)
    if p.is_absolute():
        return str(p)

    # CSV paths are expected relative to DATASET_ROOT, e.g.
    # DLC-2021/or/.../clip.mp4 or CCD/.../clip.mp4.
    parts = list(p.parts)
    if parts and parts[0].lower() == 'dataset':
        p = Path(*parts[1:])
    return str((DATASET_ROOT / p).resolve())


def load_stage1_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(
            f'CSV not found: {path}\n'
            f'Put train.csv / val.csv / test.csv in {DATASET_ROOT / "stage1_splits"} '
            f'or directly under {DATASET_ROOT}.'
        )

    frame = pd.read_csv(path).copy()
    required = {'video_path', 'label'}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {missing}')

    raw_paths = frame['video_path'].astype(str).str.strip()
    normalized_labels = frame['label'].astype(str).str.strip().str.upper()
    mapped = normalized_labels.map(LABEL_ALIASES)
    if mapped.isna().any():
        bad = sorted(normalized_labels[mapped.isna()].unique().tolist())
        raise ValueError(f'{path.name} has unsupported Stage 1 labels: {bad}')
    frame['label'] = mapped

    if 'video_id' not in frame.columns:
        frame['video_id'] = raw_paths.map(_stable_video_id)
    else:
        frame['video_id'] = frame['video_id'].astype(str).str.strip()

    if 'dataset' not in frame.columns:
        frame['dataset'] = raw_paths.map(_infer_dataset)
    else:
        frame['dataset'] = frame['dataset'].astype(str).str.strip()

    frame['video_path'] = raw_paths.map(_resolve_video_path)

    if frame['video_id'].duplicated().any():
        dup = frame.loc[frame['video_id'].duplicated(keep=False), 'video_id'].head(10).tolist()
        raise ValueError(f'{path.name} has duplicated video_id values: {dup}')

    missing_files = [p for p in frame['video_path'] if not Path(p).is_file()]
    if missing_files:
        raise FileNotFoundError(
            f'{path.name}: {len(missing_files)} video file(s) do not exist. '
            f'First examples: {missing_files[:3]}'
        )

    if frame['label'].nunique() < 2:
        raise ValueError(f'{path.name} must contain both ORIGINAL and RERECORDED.')

    return frame.reset_index(drop=True)


train_df = load_stage1_csv(TRAIN_CSV)
val_df = load_stage1_csv(VAL_CSV)

video_overlap = set(train_df['video_id']) & set(val_df['video_id'])
if video_overlap:
    raise ValueError(f'train/val video_id leakage: {sorted(video_overlap)[:5]}')

# If the team provides source_video_id (recommended once CCD physical pairs exist),
# perform a lightweight leakage safety check without generating any split here.
if 'source_video_id' in train_df.columns and 'source_video_id' in val_df.columns:
    train_sources = set(train_df['source_video_id'].dropna().astype(str).str.strip()) - {''}
    val_sources = set(val_df['source_video_id'].dropna().astype(str).str.strip()) - {''}
    source_overlap = train_sources & val_sources
    if source_overlap:
        raise ValueError(f'train/val source_video_id leakage: {sorted(source_overlap)[:5]}')

print('train:', len(train_df), train_df['label'].value_counts().to_dict())
print('val  :', len(val_df), val_df['label'].value_counts().to_dict())
print('datasets(train):', train_df['dataset'].value_counts().to_dict())
print('datasets(val)  :', val_df['dataset'].value_counts().to_dict())
print('test.csv is intentionally reserved for final holdout evaluation.')

## 5. Model

In [ ]:
model = build_stage1_model(
    MODEL_NAME,
    finetune_mode=CONFIG['model']['finetune_mode'],
    unfreeze_last_n=int(CONFIG['model']['unfreeze_last_n']),
    **CONFIG['model']['params'],
)
print('blocks:', len(model.blocks), '| feature dim:', model.feature_dim)
print('parameters:', count_parameters(model))
print('load report:', getattr(model, 'load_report', None))

### 5.1 Datasets and loaders

In [ ]:
video_config = CONFIG['data']
augmentation_config = CONFIG['augmentation']
preprocessing = model.preprocessing()
print('checkpoint preprocessing:', preprocessing)

train_transform, val_transform = build_video_transforms(
    crop_size=int(preprocessing['input_size']),
    mean=tuple(preprocessing['mean']),
    std=tuple(preprocessing['std']),
    train_config=ClipAugmentConfig(
        crop_size=int(preprocessing['input_size']),
        scale_range=tuple(augmentation_config['scale_range']),
        ratio_range=tuple(augmentation_config['ratio_range']),
        hflip_prob=float(augmentation_config['hflip_prob']),
        brightness=float(augmentation_config['brightness']),
        contrast=float(augmentation_config['contrast']),
        perspective_prob=float(augmentation_config['perspective_prob']),
        perspective_scale=float(augmentation_config['perspective_scale']),
    ),
)

train_dataset = Stage1VideoDataset(
    train_df,
    clip_sampler=build_clip_sampler(train=True, num_frames=int(video_config['num_frames']), strides=video_config['train_strides'], num_clips=int(video_config['train_num_clips'])),
    transform=train_transform, on_error='zero', deterministic=False,
)
val_dataset = Stage1VideoDataset(
    val_df,
    clip_sampler=build_clip_sampler(train=False, num_frames=int(video_config['num_frames']), val_stride=int(video_config['val_stride']), num_clips=int(video_config['val_num_clips'])),
    transform=val_transform, on_error='zero', deterministic=True,
)
train_loader = build_dataloader(train_dataset, batch_size=int(video_config['batch_size']), shuffle=True, num_workers=int(video_config['num_workers']), seed=SEED, drop_last=True)
val_loader = build_dataloader(val_dataset, batch_size=int(video_config['val_batch_size']), shuffle=False, num_workers=int(video_config['num_workers']), seed=SEED)

batch = next(iter(train_loader))
print('clip batch:', tuple(batch['pixels'].shape), '| labels:', batch['label'].tolist())

## 6. Training

In [ ]:
train_config = CONFIG['train']
trainer_config = TrainConfig(
    epochs=int(train_config['epochs']),
    learning_rate=float(train_config['learning_rate']),
    head_learning_rate=(
        float(train_config['head_learning_rate'])
        if train_config.get('head_learning_rate') is not None
        else None
    ),
    weight_decay=float(train_config['weight_decay']),
    warmup_ratio=float(train_config['warmup_ratio']),
    grad_accum_steps=int(train_config['grad_accum_steps']),
    max_grad_norm=float(train_config['max_grad_norm']),
    amp=bool(train_config['amp']),
    label_smoothing=float(train_config.get('label_smoothing', 0.0)),
    early_stopping_patience=int(train_config['early_stopping_patience']),
    eval_every=int(train_config['eval_every']),
    seed=SEED,
    output_dir=RUN_DIR,
    model_name=MODEL_NAME,
    wandb_enabled=False,
)

trainer = Stage1Trainer(
    model,
    trainer_config,
    adapter=ADAPTER,
    aggregation=AggregationConfig(
        frame_method=CONFIG['evaluation']['aggregation']['frame_method'],
        video_method=CONFIG['evaluation']['aggregation']['video_method'],
    ),
    model_config={'name': MODEL_NAME, 'params': CONFIG['model']['params']},
)

print('device:', trainer.device, '| amp:', trainer.amp)
outcome = trainer.fit(train_loader, val_loader)
print(
    f'best epoch {outcome.best_epoch}: Macro-F1 {outcome.best_macro_f1:.4f} '
    f'at threshold {outcome.best_threshold:.3f}'
)

## 7. Validation & save

In [ ]:
load_checkpoint(
    RUN_DIR / 'best.pt',
    model=model,
    map_location=trainer.device,
    restore_rng_state=False,
)

evaluator = Stage1Evaluator(
    model,
    ADAPTER,
    device=trainer.device,
    amp=trainer.amp,
    aggregation=trainer.aggregation,
)
result, units = evaluator.evaluate(val_loader, return_units=True)

print(f'Macro-F1            : {result.macro_f1:.4f}')
print(f'Macro-F1 @ thr 0.5  : {result.macro_f1_at_default:.4f}')
print(f'optimal threshold   : {result.threshold:.4f}')
print(f'class-wise F1       : {result.per_class_f1}')
print(f'per-dataset Macro-F1: {result.dataset_scores}')
print(f'videos              : {result.num_videos} ({result.num_invalid_videos} with decode problems)')

save_predictions(result.predictions, RUN_DIR / 'val_predictions.csv')
outcome.history.to_csv(RUN_DIR / 'history.csv', index=False)
summary = {
    'model_name': MODEL_NAME,
    'val_macro_f1': float(result.macro_f1),
    'val_macro_f1_at_0.5': float(result.macro_f1_at_default),
    'best_threshold': float(result.threshold),
    'per_class_f1': result.per_class_f1,
    'best_epoch': int(outcome.best_epoch),
    'num_val_videos': int(result.num_videos),
    'preprocessing': dict(model.preprocessing()),
}
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
print('saved to:', RUN_DIR)